In [1]:
import pandas as pd

In [3]:
file_path = "../data/interim/transformed_online_retail.xlsx"
sales = pd.read_excel(file_path)
sales

,Unnamed: 0,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,Day,Hour,Dayofweek,DayName,MonthName
0,0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,2010,12,1,8,2,Wednesday,December
1,1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010,12,1,8,2,Wednesday,December
2,2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,2010,12,1,8,2,Wednesday,December
3,3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010,12,1,8,2,Wednesday,December
4,4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010,12,1,8,2,Wednesday,December
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392687,541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,2011,12,9,12,4,Friday,December
392688,541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,2011,12,9,12,4,Friday,December
392689,541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,2011,12,9,12,4,Friday,December
392690,541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,2011,12,9,12,4,Friday,December


In [4]:
sales["TotalAmount"] = sales["Quantity"] * sales["UnitPrice"]

In [5]:
sales[["Quantity","UnitPrice","TotalAmount"]].head(10)

,Quantity,UnitPrice,TotalAmount
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34
5,2,7.65,15.30
6,6,4.25,25.50
7,6,1.85,11.10
8,6,1.85,11.10
9,32,1.69,54.08


In [6]:
sales["Isweekend"] = sales["Dayofweek"] >= 5

In [7]:
def get_time_period(hour):
    if hour < 12:
        return "morning"
    elif hour < 17:
        return "afternoon"
    else:
        return "evening"
sales["TimePeriod"] = sales["Hour"].apply(get_time_period)

In [8]:
customer_spending = sales.groupby("CustomerID")["TotalAmount"].sum().rename("CustomerTotalSpend")
customer_spending

CustomerID
12346    77183.60
12347     4310.00
12348     1797.24
12349     1757.55
12350      334.40
           ...   
18280      180.60
18281       80.82
18282      178.05
18283     2045.53
18287     1837.28
Name: CustomerTotalSpend, Length: 4338, dtype: float64

In [9]:
sales = sales.merge(
    customer_spending,
    on= "CustomerID",
    how= "left"
)
sales

,Unnamed: 0,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,Day,Hour,Dayofweek,DayName,MonthName,TotalAmount,Isweekend,TimePeriod,CustomerTotalSpend
0,0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,2010,12,1,8,2,Wednesday,December,15.30,False,morning,5391.21
1,1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010,12,1,8,2,Wednesday,December,20.34,False,morning,5391.21
2,2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,2010,12,1,8,2,Wednesday,December,22.00,False,morning,5391.21
3,3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010,12,1,8,2,Wednesday,December,20.34,False,morning,5391.21
4,4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010,12,1,8,2,Wednesday,December,20.34,False,morning,5391.21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392687,541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,2011,12,9,12,4,Friday,December,10.20,False,afternoon,862.81
392688,541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,2011,12,9,12,4,Friday,December,12.60,False,afternoon,862.81
392689,541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,2011,12,9,12,4,Friday,December,16.60,False,afternoon,862.81
392690,541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,2011,12,9,12,4,Friday,December,16.60,False,afternoon,862.81


In [10]:
customer_frequnecy = sales.groupby("CustomerID")["InvoiceNo"].nunique().rename("CustomerPurchaseFrequency")
customer_frequnecy



CustomerID
12346     1
12347     7
12348     4
12349     1
12350     1
         ..
18280     1
18281     1
18282     2
18283    16
18287     3
Name: CustomerPurchaseFrequency, Length: 4338, dtype: int64

In [11]:
sales = sales.merge(
    customer_frequnecy,
    on="CustomerID",
    how="left"
)

In [12]:
sales["CustomerAvgOrderValue"] = (
    sales["CustomerTotalSpend"] / sales["CustomerPurchaseFrequency"]
    )


In [13]:
sales

,Unnamed: 0,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,...,Hour,Dayofweek,DayName,MonthName,TotalAmount,Isweekend,TimePeriod,CustomerTotalSpend,CustomerPurchaseFrequency,CustomerAvgOrderValue
0,0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,2010,...,8,2,Wednesday,December,15.30,False,morning,5391.21,34,158.5650
1,1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010,...,8,2,Wednesday,December,20.34,False,morning,5391.21,34,158.5650
2,2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,2010,...,8,2,Wednesday,December,22.00,False,morning,5391.21,34,158.5650
3,3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010,...,8,2,Wednesday,December,20.34,False,morning,5391.21,34,158.5650
4,4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010,...,8,2,Wednesday,December,20.34,False,morning,5391.21,34,158.5650
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392687,541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,2011,...,12,4,Friday,December,10.20,False,afternoon,862.81,4,215.7025
392688,541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,2011,...,12,4,Friday,December,12.60,False,afternoon,862.81,4,215.7025
392689,541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,2011,...,12,4,Friday,December,16.60,False,afternoon,862.81,4,215.7025
392690,541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,2011,...,12,4,Friday,December,16.60,False,afternoon,862.81,4,215.7025


In [14]:
sales.to_excel("../data/interim/featured_online_retail.xlsx")